In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import random
import numpy as np

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [3]:
class ResidualBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_c)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class ScalableResNetLite(nn.Module):
    def __init__(self, channels=64, depth=3, num_classes=10):
        super().__init__()

        self.conv1 = nn.Conv2d(3, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)

        layers = []
        c = channels
        for i in range(depth):
            stride = 1 if i == 0 else 2
            out_c = c if i == 0 else c * 2
            layers.append(ResidualBlock(c, out_c, stride))
            c = out_c

        self.res_layers = nn.Sequential(*layers)
        self.linear = nn.Linear(c, num_classes)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.res_layers(out)
        out = F.avg_pool2d(out, out.size()[3])
        out = out.view(out.size(0), -1)
        return self.linear(out)

In [4]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
])

train_dataset_full = datasets.CIFAR10(root="../datasets", train=True,
                                      download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root="../datasets", train=False,
                                download=True, transform=transform_test)

val_size = 5000
train_size = len(train_dataset_full) - val_size
train_dataset, val_dataset = random_split(train_dataset_full, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

c:\Users\getro\OneDrive\Desktop\School\e_graduate\Advanced Machine Learning\final-project\gap\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [5]:
def make_optimizer(opt_name, model, lr, wd):
    if opt_name == "sgd":
        return optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    else:
        return optim.Adam(model.parameters(), lr=lr, weight_decay=wd)

In [6]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    correct, total, running_loss = 0, 0, 0.0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * x.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return correct / total, running_loss / total

In [7]:
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total

In [8]:
lrs = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
channels_list = [64, 96, 128]
depth_list = [3, 4, 5]
opts = ["sgd", "adam"]
weight_decays = [0, 1e-4, 5e-4]

criterion = nn.CrossEntropyLoss()

coarse_results = []

for lr in lrs:
    for ch in channels_list:
        for d in depth_list:
            for opt in opts:
                for wd in weight_decays:
                    model = ScalableResNetLite(channels=ch, depth=d).to(device)
                    optimizer = make_optimizer(opt, model, lr, wd)

                    train_acc, _ = train_one_epoch(model, train_loader, criterion, optimizer)
                    val_acc = evaluate(model, val_loader)

                    coarse_results.append(
                        ({"lr": lr, "channels": ch, "depth": d, "opt": opt, "wd": wd},
                         val_acc)
                    )
                    print("SA:", lr, ch, d, opt, wd, "val_acc=", val_acc)

coarse_results.sort(key=lambda x: x[1], reverse=True)

SA: 0.0001 64 3 sgd 0 val_acc= 0.2864
SA: 0.0001 64 3 sgd 0.0001 val_acc= 0.2826
SA: 0.0001 64 3 sgd 0.0005 val_acc= 0.2742
SA: 0.0001 64 3 adam 0 val_acc= 0.5124
SA: 0.0001 64 3 adam 0.0001 val_acc= 0.5094
SA: 0.0001 64 3 adam 0.0005 val_acc= 0.4794
SA: 0.0001 64 4 sgd 0 val_acc= 0.3042
SA: 0.0001 64 4 sgd 0.0001 val_acc= 0.2994
SA: 0.0001 64 4 sgd 0.0005 val_acc= 0.2876
SA: 0.0001 64 4 adam 0 val_acc= 0.5624
SA: 0.0001 64 4 adam 0.0001 val_acc= 0.5688
SA: 0.0001 64 4 adam 0.0005 val_acc= 0.5626
SA: 0.0001 64 5 sgd 0 val_acc= 0.3124
SA: 0.0001 64 5 sgd 0.0001 val_acc= 0.3202
SA: 0.0001 64 5 sgd 0.0005 val_acc= 0.313
SA: 0.0001 64 5 adam 0 val_acc= 0.5516
SA: 0.0001 64 5 adam 0.0001 val_acc= 0.5538
SA: 0.0001 64 5 adam 0.0005 val_acc= 0.5296
SA: 0.0001 96 3 sgd 0 val_acc= 0.2938
SA: 0.0001 96 3 sgd 0.0001 val_acc= 0.3018
SA: 0.0001 96 3 sgd 0.0005 val_acc= 0.2958
SA: 0.0001 96 3 adam 0 val_acc= 0.5454
SA: 0.0001 96 3 adam 0.0001 val_acc= 0.5322
SA: 0.0001 96 3 adam 0.0005 val_acc= 0.50

In [9]:
def derive_refined_space(coarse_results, top_frac=0.2):
    top_k = max(1, int(len(coarse_results) * top_frac))
    top_configs = [cfg for (cfg, acc) in coarse_results[:top_k]]

    lrs = [c["lr"] for c in top_configs]
    channels = [c["channels"] for c in top_configs]
    depths = [c["depth"] for c in top_configs]
    opts = [c["opt"] for c in top_configs]
    wds = [c["wd"] for c in top_configs]

    refined_space = {
        "lr": (min(lrs), max(lrs)),
        "channels": sorted(set(channels)),
        "depth": sorted(set(depths)),
        "opt": sorted(set(opts)),
        "wd": sorted(set(wds)),
    }
    return refined_space

refined_space = derive_refined_space(coarse_results)
print("Refined space:", refined_space)

Refined space: {'lr': (0.0001, 0.01), 'channels': [64, 96, 128], 'depth': [3, 4, 5], 'opt': ['adam', 'sgd'], 'wd': [0, 0.0001, 0.0005]}


In [10]:
def sample_config(space):
    lr_min, lr_max = space["lr"]
    lr = 10 ** random.uniform(np.log10(lr_min), np.log10(lr_max))
    ch = random.choice(space["channels"])
    d = random.choice(space["depth"])
    opt = random.choice(space["opt"])
    wd = random.choice(space["wd"])
    return {"lr": lr, "channels": ch, "depth": d, "opt": opt, "wd": wd}

In [11]:
num_trials = 30
rand_results = []

for t in range(num_trials):
    config = sample_config(refined_space)
    model = ScalableResNetLite(config["channels"], config["depth"]).to(device)
    optimizer = make_optimizer(config["opt"], model, config["lr"], config["wd"])

    train_acc, _ = train_one_epoch(model, train_loader, criterion, optimizer)
    val_acc = evaluate(model, val_loader)

    rand_results.append((config, val_acc))
    print("RS trial", t, config, "val_acc=", val_acc)

rand_results.sort(key=lambda x: x[1], reverse=True)
best_config, best_val = rand_results[0]
print("Best config:", best_config, "val_acc=", best_val)

RS trial 0 {'lr': np.float64(0.0011141057826270758), 'channels': 128, 'depth': 4, 'opt': 'sgd', 'wd': 0.0001} val_acc= 0.5056
RS trial 1 {'lr': np.float64(0.002272824480759304), 'channels': 96, 'depth': 3, 'opt': 'sgd', 'wd': 0.0001} val_acc= 0.5132
RS trial 2 {'lr': np.float64(0.0004963141503080228), 'channels': 128, 'depth': 3, 'opt': 'adam', 'wd': 0.0001} val_acc= 0.5944
RS trial 3 {'lr': np.float64(0.00016729234685242532), 'channels': 128, 'depth': 4, 'opt': 'sgd', 'wd': 0.0005} val_acc= 0.36
RS trial 4 {'lr': np.float64(0.008621953894180306), 'channels': 64, 'depth': 5, 'opt': 'sgd', 'wd': 0.0005} val_acc= 0.4246
RS trial 5 {'lr': np.float64(0.00043498804429893526), 'channels': 96, 'depth': 5, 'opt': 'sgd', 'wd': 0} val_acc= 0.4796
RS trial 6 {'lr': np.float64(0.0002780229736102776), 'channels': 128, 'depth': 3, 'opt': 'adam', 'wd': 0} val_acc= 0.487
RS trial 7 {'lr': np.float64(0.0017690069787564889), 'channels': 64, 'depth': 5, 'opt': 'sgd', 'wd': 0} val_acc= 0.503
RS trial 8 {'

In [12]:
def cross_val_score(config, dataset, k=3):
    fold_size = len(dataset) // k
    indices = list(range(len(dataset)))
    random.shuffle(indices)
    scores = []

    for i in range(k):
        val_idx = indices[i * fold_size:(i + 1) * fold_size]
        train_idx = indices[:i * fold_size] + indices[(i + 1) * fold_size:]

        train_subset = torch.utils.data.Subset(dataset, train_idx)
        val_subset = torch.utils.data.Subset(dataset, val_idx)

        train_loader_cv = DataLoader(train_subset, batch_size=128, shuffle=True, num_workers=2)
        val_loader_cv = DataLoader(val_subset, batch_size=128, shuffle=False, num_workers=2)

        model = ScalableResNetLite(config["channels"], config["depth"]).to(device)
        optimizer = make_optimizer(config["opt"], model, config["lr"], config["wd"])

        train_one_epoch(model, train_loader_cv, criterion, optimizer)
        val_acc = evaluate(model, val_loader_cv)
        scores.append(val_acc)

    return sum(scores) / len(scores)

cv_score = cross_val_score(best_config, train_dataset_full, k=3)
print("CV score for best config:", cv_score)

CV score for best config: 0.47831913276531063


In [13]:
final_model = ScalableResNetLite(best_config["channels"], best_config["depth"]).to(device)
optimizer = make_optimizer(best_config["opt"], final_model,
                           best_config["lr"], best_config["wd"])

num_epochs = 20
for epoch in range(num_epochs):
    train_acc, train_loss = train_one_epoch(final_model, train_loader, criterion, optimizer)
    val_acc = evaluate(final_model, val_loader)
    print(epoch, "train_acc=", train_acc, "val_acc=", val_acc)

test_acc = evaluate(final_model, test_loader)
print("Final test accuracy:", test_acc)

0 train_acc= 0.5106888888888889 val_acc= 0.4746
1 train_acc= 0.6665777777777778 val_acc= 0.6802
2 train_acc= 0.7344666666666667 val_acc= 0.732
3 train_acc= 0.7728666666666667 val_acc= 0.736
4 train_acc= 0.8009333333333334 val_acc= 0.7766
5 train_acc= 0.8178888888888889 val_acc= 0.7926
6 train_acc= 0.8312222222222222 val_acc= 0.8092
7 train_acc= 0.8466222222222223 val_acc= 0.8246
8 train_acc= 0.8528222222222223 val_acc= 0.8222
9 train_acc= 0.8645555555555555 val_acc= 0.8088
10 train_acc= 0.8697555555555555 val_acc= 0.839
11 train_acc= 0.8775111111111111 val_acc= 0.8352
12 train_acc= 0.8840444444444444 val_acc= 0.8408
13 train_acc= 0.8907333333333334 val_acc= 0.8632
14 train_acc= 0.8948888888888888 val_acc= 0.8512
15 train_acc= 0.8996 val_acc= 0.849
16 train_acc= 0.9037555555555555 val_acc= 0.8748
17 train_acc= 0.9080666666666667 val_acc= 0.852
18 train_acc= 0.9124888888888889 val_acc= 0.8696
19 train_acc= 0.9153111111111111 val_acc= 0.8694
Final test accuracy: 0.8619


In [14]:
torch.save(final_model.state_dict(), "../saved_models/cifar10_classifier.pth")

In [ ]:
import json
import numpy as np
from datetime import datetime
best_config = {'lr': np.float64(0.0001838809343739058), 'channels': 96, 'depth': 4, 'opt': 'adam', 'wd': 0.0005}
metadata = {
    "model_config": {
        "lr": float(best_config["lr"]),
        "channels": int(best_config["channels"]),
        "depth": int(best_config["depth"]),
        "optimizer": best_config["opt"],
        "weight_decay": float(best_config["wd"]),
    },
    "test_accuracy": 0.8619,
}


with open("../saved_models/cifar_10v2_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)
